# Rate4Site conservation comparison in Google Colab (381 records)

**Objective.** Use two input CSVs to search the full reviewed UniProtKB/Swiss-Prot database with BLASTp, align homologs with MAFFT, and estimate site-specific rates with Rate4Site `-Ib`. Compute conservation differences for two comparisons, apply one-sided Wilcoxon signed-rank tests, and export four CSVs plus a PNG and an SVG in one ZIP. Output filenames and column headers are in English.

- Of the 381 input records, 329 have a 15-mer that overlaps the DNA-binding domain (DBD). The other 52 are marked `NO_OVERLAP`.
- Upload `human-multimodal_dataset.csv` and `interpro_dbd_paper_table_human_deeptfactor.csv`. The original input CSVs can be used directly.
- An internet connection is required to download the approximately **90 MB** compressed Swiss-Prot file. The extracted FASTA and BLAST database require additional storage.
- Run a **three-protein pilot** before starting the full analysis. A free Colab session may end before the analysis finishes. Save intermediate files to Google Drive to resume later.
- Use the ZIP for publication only after the full run is complete and you have checked how many records passed all filters.

The notebook includes the complete `rate4site_pipeline.py` script, including the `db` and `search` stages. Run the cell in Section 2 to write it to Colab.


## 1. Install Linux tools and Python dependencies

A Python 3 CPU runtime in Google Colab is sufficient. Install `rate4site`, `ncbi-blast+`, and `mafft`. If the initial installation fails, the cell enables Ubuntu's `universe` repository and retries.


In [ ]:
import subprocess, shutil, sys

subprocess.run(['apt-get', 'update', '-qq'], check=True)
packages = ['mafft', 'ncbi-blast+', 'rate4site', 'wget']
result = subprocess.run(['apt-get', 'install', '-y', '-qq', *packages],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if result.returncode != 0:
    print('Initial installation failed. Trying again with the Ubuntu universe repository enabled...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'software-properties-common'], check=True)
    subprocess.run(['add-apt-repository', '-y', 'universe'], check=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', *packages], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'matplotlib'], check=True)
for command in ['mafft', 'blastp', 'makeblastdb', 'blastdbcmd', 'rate4site']:
    location = shutil.which(command)
    print(command, '=>', location)
    assert location is not None, f'{command} was not installed'
print('All required tools are installed.')

## 2. Write the pipeline script to Colab

Run the cell below to write the complete workflow to `/content/rate4site_pipeline.py`.


In [ ]:
%%writefile /content/rate4site_pipeline.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Reproducible Colab pipeline: input CSVs -> reviewed Swiss-Prot BLAST -> MAFFT -> Rate4Site -> 6-file ZIP.

This script does not fabricate residue scores. Requires online Swiss-Prot for 'db' and
Linux executables ncbi-blast+, mafft, rate4site for later stages.
All residue positions are one-based and inclusive.  'report' always includes ALL records.
"""
import argparse
import collections
import csv
import gzip
import hashlib
import json
import math
import os
import re
import shutil
import statistics
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

SWISSPROT_URL = 'https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz'
RELEASE_URL = 'https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/reldate.txt'
AA20 = set('ACDEFGHIKLMNPQRSTVWY')
# Original InterPro CSV headers are escaped for backward compatibility.
LEGACY_INTERPRO_HEADERS = {
    '\u6a21\u578b\u7247\u6bb5': 'Model Fragment',
    '\u7247\u6bb5\u4f4d\u7f6e': 'Fragment Positions',
    'InterPro\u7ed3\u6784\u57df': 'InterPro Domain',
    '\u7ed3\u6784\u57df\u4f4d\u7f6e': 'Domain Positions',
    '\u91cd\u53e0': 'Overlap',
}
FIELDS = [
    'ProteinID', 'Label', 'Model Fragment', 'Fragment Positions', 'InterPro Domain', 'Domain Positions', 'Overlap',
    'Overlap Residue Count', 'Rate4Site File', '15-mer Mean Rate (z-score)', 'DBD Background Mean Rate (z-score)',
    'Conservation Difference (Full 15-mer)', 'Overlap Mean Rate (z-score)', 'Conservation Difference (DBD Overlap)',
    'Scored 15-mer Residues', 'Scored Overlap Residues', 'Scored Background Residues', 'Status'
]
STAT_FIELDS = ['Comparison', 'Proteins Passing Filters', 'More Conserved in High-Contribution Region (n)', 'More Conserved in High-Contribution Region (%)',
               'Mean Difference', 'Median Difference', 'One-Sided Wilcoxon P-Value']
FILE_NAMES = ['Rate4Site_All_Statuses.csv', 'Rate4Site_Publication_Comparison.csv',
              'Rate4Site_Summary_Statistics.csv', 'Rate4Site_Status_Counts.csv',
              'Rate4Site_Difference_Distribution.png', 'Rate4Site_Difference_Distribution.svg']


def run(args, **kwargs):
    print('[EXEC]', ' '.join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True, **kwargs)


def read_csv(path):
    with open(path, 'r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def write_csv(path, fields, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def write_fasta(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for header, sequence in records:
            f.write('>' + header + '\n')
            for i in range(0, len(sequence), 80):
                f.write(sequence[i:i+80] + '\n')


def iter_fasta(path):
    name, pieces = None, []
    with Path(path).open(encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if name is not None:
                    yield name, ''.join(pieces).upper()
                name, pieces = line[1:].split()[0], []
            elif name is not None:
                pieces.append(line)
            else:
                raise ValueError('Not a FASTA: ' + str(path))
    if name is not None:
        yield name, ''.join(pieces).upper()


def parse_range(s):
    m = re.fullmatch(r'\s*(\d+)\s*[-–]\s*(\d+)\s*', str(s))
    if m is None:
        raise ValueError('Invalid range: ' + repr(s))
    start, end = int(m[1]), int(m[2])
    if not (1 <= start <= end):
        raise ValueError('Invalid residue interval: ' + repr(s))
    return start, end


def load_records(dataset_file, interpro_file):
    mapping = {}
    for r in read_csv(dataset_file):
        pid = r['Protein_ID'].strip()
        if pid in mapping:
            raise ValueError('Duplicated Protein_ID in dataset: ' + pid)
        seq = r['Sequence'].strip().upper()
        if len(seq) != int(r['Length']):
            raise ValueError('Sequence length inconsistent: ' + pid)
        mapping[pid] = (seq, r['Label'].strip())
    items, seen = [], set()
    for r in read_csv(interpro_file):
        for original, english in LEGACY_INTERPRO_HEADERS.items():
            if english not in r and original in r:
                r[english] = r[original]
        pid = r['ProteinID'].strip()
        if pid in seen:
            raise ValueError('Repeated ProteinID in InterPro table; requires unique record key: ' + pid)
        seen.add(pid)
        if pid not in mapping:
            raise ValueError('Missing ProteinID in dataset: ' + pid)
        seq, label = mapping[pid]
        a, b = parse_range(r['Fragment Positions'])
        c, d = parse_range(r['Domain Positions'])
        frag = r['Model Fragment'].strip().upper()
        if b > len(seq) or d > len(seq) or frag != seq[a-1:b] or len(frag) != b-a+1:
            raise ValueError('Sequence/position mismatch: ' + pid)
        overlap = max(0, min(b, d) - max(a, c) + 1)
        if r['Overlap'].strip() != f'{overlap}/{len(frag)}':
            raise ValueError('Wrong overlap: ' + pid)
        items.append({**r, 'ProteinID': pid, 'Label': label, '_seq': seq,
                      '_fs': a, '_fe': b, '_ds': c, '_de': d,
                      '_hit': list(range(max(a, c), min(b, d) + 1)) if overlap else []})
    return items


def subset(items, args):
    usable = [x for x in items if x['_hit']]
    if args.start < 0 or args.limit < 0:
        raise ValueError('--start and --limit cannot be negative')
    return usable[args.start:args.start+args.limit if args.limit else None]


def cmd_prepare(records, args):
    wd = Path(args.workdir)
    queries = wd / 'queries'
    queries.mkdir(parents=True, exist_ok=True)
    checks = []
    for r in records:
        pid = r['ProteinID']
        write_fasta(queries / f'{pid}.fasta', [(pid, r['_seq'])])
        checks.append({'ProteinID': pid, 'Label': r['Label'], 'Protein Length': len(r['_seq']),
                       'Fragment Positions': r['Fragment Positions'], 'Domain Positions': r['Domain Positions'],
                       'Verified Overlap': r['Overlap'], 'Eligible for Within-DBD Comparison': 'Yes' if r['_hit'] else 'No'})
    write_fasta(wd / 'all_queries.fasta', [(r['ProteinID'], r['_seq']) for r in records])
    write_csv(wd / 'Input_Validation_and_Task_List.csv', list(checks[0]), checks)
    print(f'[PREPARE] All {len(records)} records verified; eligible {sum(bool(x["_hit"]) for x in records)}; no overlap {sum(not x["_hit"] for x in records)}')


def sha256sum(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def cmd_db(records, args):
    dbdir = Path(args.dbdir)
    dbdir.mkdir(parents=True, exist_ok=True)
    compressed = dbdir / 'uniprot_sprot.fasta.gz'
    fasta = dbdir / 'uniprot_sprot.fasta'
    dbprefix = dbdir / 'swissprot_blast'
    old_metadata = dbdir / 'swissprot_metadata.json'
    first_download = not compressed.is_file() or compressed.stat().st_size == 0
    if first_download:
        run(['wget', '-c', '--tries=5', '--timeout=60', '-O', compressed, args.swissprot_url])
    if not fasta.is_file() or fasta.stat().st_size == 0:
        tmp = fasta.with_suffix('.fasta.tmp')
        print('[DB] Decompressing downloaded Swiss-Prot archive...')
        for attempt in range(2):
            try:
                with gzip.open(compressed, 'rb') as src, tmp.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                tmp.replace(fasta)
                break
            except (EOFError, OSError) as exc:
                tmp.unlink(missing_ok=True)
                if attempt > 0:
                    raise RuntimeError('Swiss-Prot download appears incomplete; retry db stage') from exc
                print('[DB WARNING] Damaged or partial gzip; resuming download:', exc)
                # wget -c needs the existing partial file to resume; if server does not support
                # resume or returns an inconsistent file, retry with a clean download manually.
                run(['wget', '-c', '--tries=5', '--timeout=60', '-O', compressed, args.swissprot_url])
    valid = subprocess.run([args.blastdbcmd, '-db', str(dbprefix), '-info'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
    if not valid:
        run([args.makeblastdb, '-in', fasta, '-dbtype', 'prot', '-parse_seqids', '-out', dbprefix])
    current_digest = sha256sum(compressed)
    old = json.loads(old_metadata.read_text(encoding='utf-8')) if old_metadata.exists() else {}
    if old.get('download_sha256') == current_digest:
        # Keep the release date downloaded at the same time as the compressed snapshot.
        print('[DB] Reusing cached Swiss-Prot snapshot:', current_digest)
        return
    try:
        urllib.request.urlretrieve(RELEASE_URL, str(dbdir / 'reldate.txt'))
    except Exception as exc:
        print('[DB WARNING] Could not get release date:', exc)
    meta = {'source': 'UniProtKB/Swiss-Prot reviewed', 'url': args.swissprot_url,
            'download_sha256': current_digest, 'download_utc': datetime.now(timezone.utc).isoformat(),
            'fasta': str(fasta.resolve()), 'dbprefix': str(dbprefix.resolve()),
            'release_file': str(dbdir / 'reldate.txt')}
    old_metadata.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
    print('[DB] Swiss-Prot BLAST database ready:', dbprefix)
    print('[DB] SHA256:', current_digest)


def uniprot_id(header):
    # Swiss-Prot IDs: sp|P12345|GENE_SPECIES (both FASTA and BLAST subject IDs)
    if '|' in header:
        parts = header.split('|')
        if len(parts) >= 2 and parts[0].lower() == 'sp':
            return parts[1]
    return header.split()[0]


def cmd_search(records, args):
    target = subset(records, args)
    if not target:
        print('[SEARCH] No eligible records in chosen interval')
        return
    wd = Path(args.workdir)
    out = wd / 'homologs'
    out.mkdir(parents=True, exist_ok=True)
    searchdir = wd / 'blast_search'
    searchdir.mkdir(parents=True, exist_ok=True)
    qf = searchdir / f'queries_{args.start}_{args.limit or "all"}.fasta'
    hits = searchdir / f'hits_{args.start}_{args.limit or "all"}.tsv'
    sigfile = hits.with_suffix('.json')
    write_fasta(qf, [(r['ProteinID'], r['_seq']) for r in target])
    qsig = sha256sum(qf)
    dbprefix = Path(args.dbdir) / 'swissprot_blast'
    fasta = Path(args.dbdir) / 'uniprot_sprot.fasta'
    if not fasta.exists():
        raise FileNotFoundError('Swiss-Prot FASTA not found; first run db stage')
    signature = {'query_sha256': qsig, 'database_sha256': sha256sum(Path(args.dbdir) / 'uniprot_sprot.fasta.gz'),
                 'evalue': args.evalue, 'max_target_seqs': args.max_target_seqs}
    reuse = hits.is_file() and sigfile.is_file() and json.loads(sigfile.read_text()) == signature
    if reuse:
        print('[SEARCH] Reusing matching BLAST results:', hits)
    else:
        temp = hits.with_suffix('.tsv.tmp')
        run([args.blastp, '-query', qf, '-db', dbprefix, '-out', temp,
             '-evalue', str(args.evalue), '-outfmt',
             '6 qseqid sseqid pident length qlen slen qcovs evalue bitscore',
             '-max_target_seqs', str(args.max_target_seqs), '-max_hsps', '1',
             '-num_threads', str(args.threads)])
        temp.replace(hits)
        sigfile.write_text(json.dumps(signature, indent=2), encoding='utf-8')
    grouped = collections.defaultdict(list)
    req = set()
    for line in hits.read_text().splitlines():
        parts = line.split('\t')
        if len(parts) != 9:
            continue
        qid, subj, identity, alen, qlen, slen, coverage, evalue, bitscore = parts
        acc = uniprot_id(subj)
        identity, coverage, qlen, slen = float(identity), float(coverage), int(qlen), int(slen)
        if (identity < args.min_identity or identity > args.max_identity or
            coverage < args.min_query_coverage or qlen < 1 or
            not (args.min_length_ratio <= slen / qlen <= args.max_length_ratio)):
            continue
        if acc == qid:
            continue
        grouped[qid].append((acc, float(bitscore), identity, coverage))
        req.add(acc)
    print('[SEARCH] Eligible hits before full-sequence validation:', sum(map(len, grouped.values())))
    seqs = {}
    for header, seq in iter_fasta(fasta):
        acc = uniprot_id(header)
        if acc in req:
            seqs[acc] = seq
    report = []
    for r in target:
        pid = r['ProteinID']
        used_seq = {r['_seq']}
        selected = []
        for acc, score, ident, cover in sorted(grouped[pid], key=lambda x: (-x[1], x[0])):
            seq = seqs.get(acc, '')
            if not seq or set(seq) - AA20 or seq in used_seq:
                continue
            used_seq.add(seq)
            selected.append((acc, seq))
            if len(selected) >= args.max_homologs:
                break
        write_fasta(out / f'{pid}.fasta', selected)
        n = len(selected)
        report.append({'ProteinID': pid, 'BLAST Candidate Count': len(grouped[pid]),
                       'Unique Homolog Count': n, 'Total Sequences Including Reference': n+1,
                       'Meets Minimum MSA Sequence Count': 'Yes' if n+1 >= args.min_sequences else 'No'})
    write_csv(searchdir / f'homolog_search_report_{args.start}_{args.limit or "all"}.csv',
              list(report[0]), report)
    print(f'[SEARCH] {len(target)} proteins; {sum(x["Total Sequences Including Reference"] >= args.min_sequences for x in report)} ready for MAFFT')


def cmd_align(records, args):
    target = subset(records, args)
    wd = Path(args.workdir)
    mdir, tdir = wd / 'msa', wd / 'alignment_inputs'
    mdir.mkdir(parents=True, exist_ok=True)
    tdir.mkdir(parents=True, exist_ok=True)
    ok, skipped = 0, 0
    for i, r in enumerate(target, 1):
        pid = r['ProteinID']
        homolog = wd / 'homologs' / f'{pid}.fasta'
        dest = mdir / f'{pid}.fasta'
        if dest.is_file() and dest.stat().st_size > 0 and not args.force:
            if validate_msa(dest, pid, r['_seq'], args.min_sequences):
                ok += 1
                continue
        if not homolog.exists():
            skipped += 1
            continue
        pairs = [(pid, r['_seq'])]
        seen = {r['_seq']}
        for h, seq in iter_fasta(homolog):
            seq = seq.replace('-', '').replace('.', '')
            if seq not in seen and seq and not (set(seq) - AA20):
                seen.add(seq)
                pairs.append((f'H{len(pairs):03d}', seq))
        if len(pairs) < args.min_sequences:
            skipped += 1
            continue
        inp = tdir / f'{pid}.fasta'
        write_fasta(inp, pairs)
        tmp = dest.with_suffix('.fasta.tmp')
        with tmp.open('w', encoding='utf-8') as h:
            p = subprocess.run([args.mafft, '--auto', '--quiet', '--thread', '1', str(inp)],
                               stdout=h, stderr=subprocess.PIPE, text=True)
        if p.returncode != 0:
            print(f'[ALIGN FAILED] {pid}: {p.stderr[-180:]}')
            tmp.unlink(missing_ok=True)
            continue
        seqs = list(iter_fasta(tmp))
        ref = next((seq for name, seq in seqs if name == pid), None)
        if ref is None or ref.replace('-', '').replace('.', '') != r['_seq'] or any(len(s) != len(ref) for _, s in seqs):
            print(f'[ALIGN INVALID] {pid}: reference/length mismatch')
            tmp.unlink(missing_ok=True)
            continue
        # Remove insertion columns (query gaps): makes Rate4Site POS equal original residue index.
        retain = [j for j, x in enumerate(ref) if x != '-']
        trimmed = [(name, ''.join(s[j] for j in retain)) for name, s in seqs]
        if not validate_trimmed_msa(trimmed, pid, r['_seq'], args.min_sequences):
            print(f'[ALIGN INVALID] {pid}: cleaned MSA failed validation')
            tmp.unlink(missing_ok=True)
            continue
        write_fasta(dest, trimmed)
        tmp.unlink(missing_ok=True)
        ok += 1
        if i % 25 == 0:
            print('[ALIGN]', i, 'processed')
    print(f'[ALIGN] usable MSAs in batch {ok}/{len(target)}; insufficient or missing homologs {skipped}')


def validate_trimmed_msa(pairs, name, reference, min_count):
    return (len(pairs) >= min_count and len({s for _, s in pairs}) >= min_count and
            all(len(s) == len(reference) for _, s in pairs) and
            sum(n == name and s == reference for n,s in pairs) == 1)


def validate_msa(path, name, reference, min_count):
    try:
        return validate_trimmed_msa(list(iter_fasta(path)), name, reference, min_count)
    except Exception:
        return False


def parse_scores(path, reference):
    scores, nmis = {}, 0
    with Path(path).open(encoding='utf-8-sig', errors='replace') as f:
        for line in f:
            fields = line.split()
            if len(fields) < 3 or not fields[0].isdigit():
                continue
            pos = int(fields[0])
            aa = fields[1].upper()
            if len(aa) != 1 or not aa.isalpha() or not 1 <= pos <= len(reference):
                continue
            try:
                val = float(fields[2])
            except ValueError:
                continue
            if not math.isfinite(val):
                continue
            if reference[pos-1] != aa:
                nmis += 1
                continue
            scores[pos] = val
    return scores, nmis


def cmd_rate(records, args):
    target = subset(records, args)
    wd = Path(args.workdir)
    score_dir = wd / 'scores'
    score_dir.mkdir(parents=True, exist_ok=True)
    ok = 0
    for ix, r in enumerate(target, 1):
        pid = r['ProteinID']
        msa = wd / 'msa' / f'{pid}.fasta'
        if not msa.exists() or not validate_msa(msa, pid, r['_seq'], args.min_sequences):
            continue
        pdir = score_dir / pid
        pdir.mkdir(parents=True, exist_ok=True)
        outfile = pdir / 'normalized_grades.txt'
        if outfile.is_file() and not args.force:
            prior, _ = parse_scores(outfile, r['_seq'])
            if len(prior) >= len(r['_seq']) * args.coverage:
                ok += 1
                continue
        tmp = outfile.with_suffix('.txt.tmp')
        cmd = [args.rate4site, '-s', msa.resolve(), '-o', tmp.resolve(), '-a', pid, '-Ib']
        logpath = pdir / 'rate4site.log'
        try:
            with logpath.open('w', encoding='utf-8') as log:
                result = subprocess.run(list(map(str,cmd)), cwd=pdir, stdout=log,
                                        stderr=subprocess.STDOUT, timeout=args.rate_timeout)
            if result.returncode != 0:
                print(f'[RATE FAILED] {pid} exit={result.returncode} log={logpath}')
                tmp.unlink(missing_ok=True)
                continue
            parsed, mismatched = parse_scores(tmp, r['_seq']) if tmp.exists() else ({}, 0)
            if not parsed or mismatched > len(parsed) // 10:
                print(f'[RATE INVALID] {pid}: scores={len(parsed)} mismatches={mismatched}; see {logpath}')
                tmp.unlink(missing_ok=True)
                continue
            tmp.replace(outfile)
            ok += 1
        except subprocess.TimeoutExpired:
            tmp.unlink(missing_ok=True)
            print(f'[RATE TIMEOUT] {pid} ({args.rate_timeout} seconds)')
        if ix % 10 == 0:
            print('[RATE] processed', ix, '/', len(target), flush=True)
    print(f'[RATE] valid Rate4Site outputs for {ok}/{len(target)} selected proteins')


def summarize_one(r, args):
    out = {k: r.get(k, '') for k in FIELDS}
    hit = r['_hit']
    out.update({'Overlap Residue Count': len(hit), 'Rate4Site File': '', '15-mer Mean Rate (z-score)': '',
                'DBD Background Mean Rate (z-score)': '', 'Conservation Difference (Full 15-mer)': '',
                'Overlap Mean Rate (z-score)': '', 'Conservation Difference (DBD Overlap)': '',
                'Scored 15-mer Residues': 0, 'Scored Overlap Residues': 0, 'Scored Background Residues': 0, 'Status': ''})
    if not hit:
        out['Status'] = 'NO_OVERLAP'
        return out
    win = range(r['_fs'], r['_fe']+1)
    bg = [p for p in range(r['_ds'], r['_de']+1) if p not in set(hit)]
    if not bg:
        out['Status'] = 'NO_BACKGROUND'
        return out
    scorefile = Path(args.workdir) / 'scores' / r['ProteinID'] / 'normalized_grades.txt'
    if not scorefile.is_file():
        out['Status'] = 'NO_RATE4SITE_RESULT'
        return out
    out['Rate4Site File'] = str(scorefile.resolve())
    scores, mis = parse_scores(scorefile, r['_seq'])
    if mis and mis >= len(scores):
        out['Status'] = 'POSITION_MISMATCH'
        return out
    if not scores:
        out['Status'] = 'INVALID_RATE4SITE_FILE'
        return out
    a, b, c = [[scores[x] for x in group if x in scores] for group in (win, hit, bg)]
    out.update({'Scored 15-mer Residues': len(a), 'Scored Overlap Residues': len(b), 'Scored Background Residues': len(c)})
    coverage = (len(a)/len(win), len(b)/len(hit), len(c)/len(bg))
    if min(coverage) < args.coverage:
        out['Status'] = f'LOW_COVERAGE(win={coverage[0]:.0%},hit={coverage[1]:.0%},bg={coverage[2]:.0%})'
        return out
    winmean, hitmean, bgmean = statistics.mean(a), statistics.mean(b), statistics.mean(c)
    out.update({'15-mer Mean Rate (z-score)': winmean, 'DBD Background Mean Rate (z-score)': bgmean,
                'Conservation Difference (Full 15-mer)': bgmean-winmean,
                'Overlap Mean Rate (z-score)': hitmean, 'Conservation Difference (DBD Overlap)': bgmean-hitmean,
                'Status': 'OK'})
    return out


def make_stat(okrows, label, key):
    values = [float(r[key]) for r in okrows]
    n = len(values)
    pval = ''
    if values and any(abs(v) > 1e-12 for v in values):
        from scipy.stats import wilcoxon
        pval = float(wilcoxon(values, alternative='greater', zero_method='wilcox').pvalue)
    return {'Comparison': label, 'Proteins Passing Filters': n,
            'More Conserved in High-Contribution Region (n)': sum(v > 0 for v in values),
            'More Conserved in High-Contribution Region (%)': round(sum(v > 0 for v in values) / n * 100, 2) if n else '',
            'Mean Difference': statistics.mean(values) if n else '',
            'Median Difference': statistics.median(values) if n else '',
            'One-Sided Wilcoxon P-Value': pval}


def make_plot(okrows, outdir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    values = [float(r['Conservation Difference (Full 15-mer)']) for r in okrows]
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    if values:
        ax.hist(values, bins=min(25, max(5, int(math.sqrt(len(values))*2))), edgecolor='white', linewidth=.7)
    else:
        ax.text(.5, .5, 'No valid Rate4Site scores yet', transform=ax.transAxes, ha='center', va='center')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('DBD background mean rate minus 15-mer mean rate (z-score)')
    ax.set_ylabel('Protein count')
    ax.set_title(f'Rate4Site relative conservation (n={len(values)})')
    fig.tight_layout()
    fig.savefig(outdir / FILE_NAMES[4], dpi=180)
    fig.savefig(outdir / FILE_NAMES[5])
    plt.close(fig)


def cmd_report(records, args):
    outdir = Path(args.workdir) / 'results'
    outdir.mkdir(parents=True, exist_ok=True)
    rows = [summarize_one(r, args) for r in records]  # all 381, not the pilot subset
    ok = [r for r in rows if r['Status'] == 'OK']
    statuses = [{'Status': status, 'Protein Count': number} for status, number in
                collections.Counter(r['Status'] for r in rows).most_common()]
    stats = [make_stat(ok, 'Full 15-mer vs. DBD Background', 'Conservation Difference (Full 15-mer)'),
             make_stat(ok, 'DBD Overlap vs. DBD Background', 'Conservation Difference (DBD Overlap)')]
    write_csv(outdir / FILE_NAMES[0], FIELDS, rows)
    write_csv(outdir / FILE_NAMES[1], FIELDS, ok)
    write_csv(outdir / FILE_NAMES[2], STAT_FIELDS, stats)
    write_csv(outdir / FILE_NAMES[3], ['Status', 'Protein Count'], statuses)
    make_plot(ok, outdir)
    zipfile_path = Path(args.zip) if args.zip else Path(args.workdir) / 'Rate4Site_Conservation_Comparison_Results.zip'
    zipfile_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zipfile_path, mode='w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for name in FILE_NAMES:
            z.write(outdir / name, arcname=name)
    params = {name: value for name, value in vars(args).items() if name not in ('stage',)}
    (outdir / 'analysis_parameters.json').write_text(json.dumps({
        'method': 'Swiss-Prot BLASTp -> MAFFT reference-insertion-column trim -> Rate4Site -Ib',
        'difference_definition': 'mean(DBD excluding hit) - mean(fragment or DBD intersection)',
        'note': 'positive difference means high-contribution region is more conserved',
        'n_all_records': len(records), 'n_ok': len(ok), 'parameters': params},
        ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'[REPORT] all={len(records)}, OK={len(ok)}, zip={zipfile_path}')
    print('[REPORT] statuses:', collections.Counter(x['Status'] for x in rows))
    if not ok:
        print('[WARNING] This ZIP contains formatting placeholders only; no Rate4Site results yet.')


def main(argv=None):
    ap = argparse.ArgumentParser(description='Rate4Site Colab Swiss-Prot full workflow; stage then flags required')
    ap.add_argument('stage', choices=['prepare', 'db', 'search', 'align', 'rate', 'report'])
    ap.add_argument('--dataset', required=True)
    ap.add_argument('--interpro', required=True)
    ap.add_argument('--workdir', default='rate4site_work')
    ap.add_argument('--dbdir', default='/content/swissprot_db')
    ap.add_argument('--swissprot-url', default=SWISSPROT_URL)
    ap.add_argument('--blastp', default='blastp')
    ap.add_argument('--makeblastdb', default='makeblastdb')
    ap.add_argument('--blastdbcmd', default='blastdbcmd')
    ap.add_argument('--mafft', default='mafft')
    ap.add_argument('--rate4site', default='rate4site')
    ap.add_argument('--min-sequences', type=int, default=10)
    ap.add_argument('--min-identity', type=float, default=30.0)
    ap.add_argument('--max-identity', type=float, default=98.5)
    ap.add_argument('--min-query-coverage', type=float, default=70.0)
    ap.add_argument('--min-length-ratio', type=float, default=0.7)
    ap.add_argument('--max-length-ratio', type=float, default=1.5)
    ap.add_argument('--max-homologs', type=int, default=40)
    ap.add_argument('--max-target-seqs', type=int, default=300)
    ap.add_argument('--evalue', type=float, default=1e-5)
    ap.add_argument('--threads', type=int, default=2)
    ap.add_argument('--rate-timeout', type=int, default=3600)
    ap.add_argument('--coverage', type=float, default=0.8)
    ap.add_argument('--start', type=int, default=0, help='start index among 329 eligible records, for search/align/rate')
    ap.add_argument('--limit', type=int, default=0, help='max items from start; 0 means all eligible records')
    ap.add_argument('--force', action='store_true', help='rebuild an existing MSA/Rate4Site result')
    ap.add_argument('--zip', default=None)
    args = ap.parse_args(argv)
    if not 0 < args.coverage <= 1 or args.min_sequences < 3 or args.max_homologs < 2 or args.threads < 1:
        ap.error('invalid coverage/min-sequences/max-homologs/threads')
    records = load_records(args.dataset, args.interpro)
    stages = {'prepare': cmd_prepare, 'db': cmd_db, 'search': cmd_search,
              'align': cmd_align, 'rate': cmd_rate, 'report': cmd_report}
    stages[args.stage](records, args)


if __name__ == '__main__':
    main()


## 3. Choose where to save your work

Google Drive is recommended for preserving intermediate files and results across Colab sessions. The Swiss-Prot BLAST database stays in `/content` for faster searches. After a runtime reset, download Swiss-Prot again and rebuild the database.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True  # Set to False to work without Google Drive access.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/Rate4Site_SwissProt_381')
else:
    PROJECT = Path('/content/Rate4Site_SwissProt_381')

INPUT = PROJECT / 'inputs'
WORK = PROJECT / 'work'
DBDIR = Path('/content/swissprot_db')   # Fast temporary storage. Download again after a runtime reset.
SCRIPT = Path('/content/rate4site_pipeline.py')
for path in [PROJECT, INPUT, WORK, DBDIR]:
    path.mkdir(parents=True, exist_ok=True)
print('Project directory:', PROJECT)

### Record software versions

The cell below records BLAST+, MAFFT, and Rate4Site version information in `software_versions.txt` in the project folder for reproducibility.


In [ ]:
version_commands = [
    ['blastp', '-version'], ['makeblastdb', '-version'],
    ['mafft', '--version'], ['rate4site', '-h']
]
version_lines = []
for command in version_commands:
    proc = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    version_lines.append('$ ' + ' '.join(command) + '\n' + proc.stdout[:500])
(PROJECT / 'software_versions.txt').write_text('\n'.join(version_lines), encoding='utf-8')
print('Software versions saved to:', PROJECT / 'software_versions.txt')


## 4. Upload the two input CSVs

The notebook reuses both CSVs if they are already in the project's `inputs` folder. To replace them with new files of the same name, delete the cached copies before running this cell.


In [ ]:
from google.colab import files

DATASET = INPUT / 'human-multimodal_dataset.csv'
INTERPRO = INPUT / 'interpro_dbd_paper_table_human_deeptfactor.csv'
if not (DATASET.exists() and INTERPRO.exists()):
    uploaded = files.upload()
    for filename, content in uploaded.items():
        if filename in (DATASET.name, INTERPRO.name):
            (INPUT / filename).write_bytes(content)
assert DATASET.is_file(), 'Missing input file: human-multimodal_dataset.csv'
assert INTERPRO.is_file(), 'Missing input file: interpro_dbd_paper_table_human_deeptfactor.csv'
print('Input ready:', DATASET.name, DATASET.stat().st_size, 'bytes')
print('Input ready:', INTERPRO.name, INTERPRO.stat().st_size, 'bytes')

## 5. Configure commands and analysis parameters

Record these parameters for the methods section. The default BLASTp thresholds are starting points. Adjust them as needed based on homolog counts, coverage, and biological relevance. If you change the parameters, regenerate any affected results.


In [ ]:
import sys, subprocess, json

COMMON = [
    '--dataset', str(DATASET), '--interpro', str(INTERPRO),
    '--workdir', str(WORK), '--dbdir', str(DBDIR),
    '--min-sequences', '10',       # Includes the target sequence. Build an MSA only with at least 10 sequences.
    '--min-identity', '30',        # Minimum BLAST alignment identity of 30%.
    '--max-identity', '98.5',      # Filter out nearly identical sequences.
    '--min-query-coverage', '70', # Minimum reference sequence coverage of 70%.
    '--min-length-ratio', '0.7',
    '--max-length-ratio', '1.5',
    '--max-homologs', '40',
    '--max-target-seqs', '300',
    '--evalue', '1e-5',
    '--threads', '2',
    '--coverage', '0.8'           # Require valid scores for at least 80% of residues in each region.
]

def run_stage(stage, start=None, limit=None, **extra):
    command = [sys.executable, str(SCRIPT), stage, *COMMON]
    if start is not None:
        command += ['--start', str(start)]
    if limit is not None:
        command += ['--limit', str(limit)]
    for key, val in extra.items():
        option = '--' + key.replace('_', '-')
        if val is True:
            command += [option]
        elif val is not False and val is not None:
            command += [option, str(val)]
    print('Running stage:', stage, '| start =', start, '| limit =', limit, flush=True)
    subprocess.run(command, check=True)

print('Pipeline stages: prepare -> db -> search -> align -> rate -> report')

## 6. Validate the inputs and extract FASTA sequences

This step checks that each fragment matches the sequence at its reported residue positions. For the expected input data, look for `381 records verified; eligible 329; no overlap 52`.


In [ ]:
run_stage('prepare')

## 7. Download the reviewed Swiss-Prot FASTA and build a local BLAST database

This step downloads approximately 90 MB, extracts several hundred megabytes of FASTA data, and builds a local BLAST database. Rerunning the cell within the same Colab runtime reuses the database.

The release information and the downloaded file's SHA-256 checksum are saved in `reldate.txt` and `swissprot_metadata.json`, respectively. Record the specific Swiss-Prot release in the methods section.


In [ ]:
run_stage('db')
import shutil
for name in ['swissprot_metadata.json', 'reldate.txt']:
    src = DBDIR / name
    if src.exists():
        shutil.copy2(src, PROJECT / name) # Keep release metadata with the project for reproducibility.
print((DBDIR / 'swissprot_metadata.json').read_text()[:900])

## 8. Run a three-protein pilot

Use the pilot to verify that BLASTp, MAFFT, and Rate4Site run successfully. You can increase `PILOT_N` from 3 to 5 or 10. Some pilot proteins may have too few homologs to build an alignment.


In [ ]:
PILOT_N = 3
run_stage('search', start=0, limit=PILOT_N)
run_stage('align', start=0, limit=PILOT_N)
run_stage('rate', start=0, limit=PILOT_N)
run_stage('report')

## 9. Inspect the pilot results

The status table includes all 381 input records. The publication comparison table includes only records with Rate4Site scores that meet the coverage threshold. If `OK=0`, inspect `blast_search/`, `homologs/`, `msa/`, and `scores/<ProteinID>/rate4site.log`.


In [ ]:
import pandas as pd
from IPython.display import display, Image

STATUS = WORK / 'results' / 'Rate4Site_Status_Counts.csv'
SUMMARY = WORK / 'results' / 'Rate4Site_Summary_Statistics.csv'
display(pd.read_csv(STATUS, encoding='utf-8-sig'))
display(pd.read_csv(SUMMARY, encoding='utf-8-sig'))
print('Pilot ZIP:', WORK / 'Rate4Site_Conservation_Comparison_Results.zip')

## 10. Run the full analysis

Set `RUN_FULL` to `True` after checking the pilot.

- `search` queries the local Swiss-Prot BLAST database for all 329 eligible proteins.
- `align` builds and validates the multiple sequence alignments (MSAs). It removes columns with gaps in the reference so Rate4Site positions match the original protein coordinates.
- `rate` processes **10 proteins per batch** and updates the ZIP after each batch. If the session disconnects, rerun this cell to resume. Existing valid scores are skipped.
- Runtime depends on CPU speed, protein length, and the number of homologs. A Colab session may end before all records are processed.
- If you change the search parameters or protein sequences, clear the affected cached alignments and scores or force those stages to rerun.


In [ ]:
RUN_FULL = False          # Set to True after the pilot runs successfully.
BATCH_SIZE = 10
START_FROM = 0           # Use 100 to start at index 100. Leave at 0 to reuse all valid scores.

if RUN_FULL:
    run_stage('search', start=0, limit=0)   # Search all 329 eligible proteins.
    run_stage('align', start=0, limit=0)
    from runpy import run_path
    # Count eligible proteins using the same input parser as the pipeline.
    load_records = run_path(str(SCRIPT))['load_records']
    eligible_n = sum(bool(record['_hit']) for record in load_records(DATASET, INTERPRO))
    print('Proteins eligible for homolog analysis:', eligible_n)
    for start in range(START_FROM, eligible_n, BATCH_SIZE):
        print(f'Rate4Site batch {start}–{min(start+BATCH_SIZE,eligible_n)-1} / {eligible_n-1}', flush=True)
        run_stage('rate', start=start, limit=BATCH_SIZE)
        run_stage('report')  # Refresh the six-file ZIP after each batch.
    print('All eligible batches have been processed. Check the OK count and status distribution.')
else:
    print('After checking the pilot results, set RUN_FULL to True and rerun this cell.')

## 11. Inspect and download the final six-file ZIP

Use the ZIP for publication after the full analysis is complete and you have reviewed the number of successful records and the reasons for exclusion. Run `files.download(...)` to download it through your browser.


In [ ]:
from google.colab import files
import zipfile
ZIP = WORK / 'Rate4Site_Conservation_Comparison_Results.zip'
assert ZIP.exists(), 'Run the report stage first.'
with zipfile.ZipFile(ZIP) as z:
    assert len(z.namelist()) == 6
    print('ZIP contents:', *z.namelist(), sep='\n- ')
display(pd.read_csv(WORK / 'results' / 'Rate4Site_Status_Counts.csv', encoding='utf-8-sig'))
print('ZIP path:', ZIP)
# Uncomment the next line to download the ZIP through your browser.
# files.download(str(ZIP))

## 12. Interpretation and limitations

**Site conservation.** Rate4Site estimates standardized relative evolutionary rates within each protein. Lower values indicate slower evolution and greater relative conservation.

**Comparison definitions.** In both comparisons, a positive difference indicates greater conservation in the high-contribution region.

- **Full 15-mer.** Subtract the mean rate of the full 15-mer from the mean rate of the DBD residues outside the overlap. For partially overlapping records, the full 15-mer includes residues outside the DBD.
- **Overlap within the DBD.** Subtract the mean rate of the residues shared by the 15-mer and the DBD from the mean rate of the remaining DBD residues. Both sets of residues lie within the DBD.

**Homolog selection.** BLASTp finds Swiss-Prot proteins with similar sequences. Establishing orthology requires further evidence. Multidomain proteins, repeated zinc fingers, and matches to other subfamilies can produce alignments that mix distinct groups. Save the `blast_search` reports and MSAs and inspect a sample manually. For publication, record the Swiss-Prot release, BLAST thresholds, alignment and Rate4Site software versions, success rate, and reasons for exclusion.

**Inference and statistics.** Rate4Site uses empirical Bayesian inference with `-Ib`. The Wilcoxon signed-rank test uses one paired difference per protein. Statistical significance alone does not establish a causal effect on function.
